<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/movierecommadation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import html

movies = pd.read_csv('/content/movies.csv',sep=',',usecols=range(2),encoding='ISO-8859-1')
ratings=pd.read_csv('/content/ratings.csv',sep=',',usecols=range(3),encoding='ISO-8859-1')

ratings=pd.merge(movies,ratings)
ratings['title'] = ratings['title'].apply(html.unescape)
ratings.head()

,movieId,title,userId,rating
0,1,Toy Story (1995),7,3.0
1,1,Toy Story (1995),9,4.0
2,1,Toy Story (1995),13,5.0
3,1,Toy Story (1995),15,2.0
4,1,Toy Story (1995),19,3.0


In [5]:
import html

movieRatings=ratings.pivot_table(index='userId',columns='title',values='rating')
movieRatings.head()

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),Â¡Three Amigos! (1986),Ã nous la libertÃ© (Freedom for Us) (1931),Ä°tirazÄ±m Var (2014)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
Darknight=movieRatings['Toy Story (1995)']
Darknight.head()

,Toy Story (1995)
userId,
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN


In [7]:
similarMovies=movieRatings.corrwith(Darknight)   #getting the correlation for better recommadation
similarMovies=similarMovies.dropna()
df=pd.DataFrame(similarMovies)
df.head(10)
similarMovies = similarMovies.sort_values(ascending=False)
print(similarMovies)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


title
Blue Valentine (2010)                                               1.0
Boat Trip (2003)                                                    1.0
Body Double (1984)                                                  1.0
Body of Evidence (1993)                                             1.0
Zombie (a.k.a. Zombie 2: The Dead Are Among Us) (Zombi 2) (1979)    1.0
                                                                   ... 
Your Sister's Sister (2011)                                        -1.0
Margaret (2011)                                                    -1.0
Star Trek 3 (2016)                                                 -1.0
Nekromantik (1987)                                                 -1.0
Wit (2001)                                                         -1.0
Length: 4709, dtype: float64


In [8]:
import numpy as np

moviestats = ratings.groupby('title').agg({
    'rating': [np.size, np.mean]
})
moviestats.columns = ['size', 'mean']

moviestats.tail()


/tmp/ipykernel_1138/3867075094.py:3: FutureWarning: The provided callable <function mean at 0x7e11da121940> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  moviestats = ratings.groupby('title').agg({


,size,mean
title,,
xXx (2002),23,2.478261
xXx: State of the Union (2005),1,1.000000
Â¡Three Amigos! (1986),31,3.258065
Ã nous la libertÃ© (Freedom for Us) (1931),1,4.500000
Ä°tirazÄ±m Var (2014),1,3.500000


In [9]:
popularmovies=moviestats['size']>=100
moviestats[popularmovies].sort_values('mean',ascending=False)[:15]

,size,mean
title,,
"Godfather, The (1972)",200,4.487500
"Shawshank Redemption, The (1994)",311,4.487138
"Godfather: Part II, The (1974)",135,4.385185
"Usual Suspects, The (1995)",201,4.370647
Schindler's List (1993),244,4.303279
One Flew Over the Cuckoo's Nest (1975),144,4.256944
Fargo (1996),224,4.256696
Pulp Fiction (1994),324,4.256173
American Beauty (1999),220,4.236364


In [10]:
df = moviestats[popularmovies].join(
    pd.DataFrame(similarMovies, columns=['similarity'])
)

df.head()   #first it checks of the +1 correlation(similarMovies) and then it sees the highest people watched

,size,mean,similarity
title,,,
2001: A Space Odyssey (1968),123,3.886179,-0.023609
Ace Ventura: Pet Detective (1994),175,2.871429,0.165157
Airplane! (1980),106,3.820755,0.285379
Aladdin (1992),215,3.674419,0.461749
Alien (1979),127,3.988189,0.355815


In [11]:
df.sort_values(['similarity'],ascending=False)[:15]

,size,mean,similarity
title,,,
Toy Story (1995),247,3.872470,1.000000
Toy Story 2 (1999),125,3.844000,0.743352
"Bug's Life, A (1998)",105,3.609524,0.677299
"Monsters, Inc. (2001)",130,3.884615,0.549582
"Dark Knight, The (2008)",121,4.235537,0.540978
Finding Nemo (2003),122,3.803279,0.537958
Austin Powers: The Spy Who Shagged Me (1999),112,3.272321,0.519847
"Lion King, The (1994)",200,3.777500,0.517524
Spider-Man (2002),134,3.522388,0.512995


Fresh ReBuilding


In [2]:
import pandas as pd
import html

movies = pd.read_csv('/content/movies.csv',sep=',',usecols=range(2),encoding='ISO-8859-1')
ratings=pd.read_csv('/content/ratings.csv',sep=',',usecols=range(3),encoding='ISO-8859-1')

ratings=pd.merge(movies,ratings)
ratings['title'] = ratings['title'].apply(html.unescape)
ratings.head()

,movieId,title,userId,rating
0,1,Toy Story (1995),7,3.0
1,1,Toy Story (1995),9,4.0
2,1,Toy Story (1995),13,5.0
3,1,Toy Story (1995),15,2.0
4,1,Toy Story (1995),19,3.0


In [3]:
movieRatings=ratings.pivot_table(index='userId',columns='title',values='rating')
movieRatings.head()

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),Â¡Three Amigos! (1986),Ã nous la libertÃ© (Freedom for Us) (1931),Ä°tirazÄ±m Var (2014)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
corrMatrix=movieRatings.corr(method='pearson',min_periods=100)
corrMatrix.head()

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),Â¡Three Amigos! (1986),Ã nous la libertÃ© (Freedom for Us) (1931),Ä°tirazÄ±m Var (2014)
title,,,,,,,,,,,,,,,,,,,,,
"""Great Performances"" Cats (1998)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
$9.99 (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Neath the Arizona Skies (1934),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
import pandas as pd

# 1. Get the ratings for User 0
# Ensure we only look at movies the user actually rated
userRatings = movieRatings.iloc[49].dropna()
# Initialize an empty Series
# Note: Always use dtype=float64 to avoid DeprecationWarnings in newer pandas versions
simCandidates = pd.Series(dtype='float64')

for i in range(0, len(userRatings.index)):
    movieName = userRatings.index[i]
    rating = userRatings.iloc[i]


    # 2. Retrieve similar movies from your correlation matrix
    if movieName in corrMatrix:
        sims = corrMatrix[movieName].dropna()

        # 3. Scale the similarity by the user's rating
        # This ensures movies similar to ones the user LIKED carry more weight
        sims = sims.map(lambda x: x * rating)
        # 4. Add the scores to the list of candidates
        simCandidates = pd.concat([simCandidates, sims])

# 5. Group by movie name and sum the scores
# (Otherwise, the same movie appears multiple times)
simCandidates = simCandidates.groupby(simCandidates.index).mean()

# 6. Sort and filter out movies the user has already seen
simCandidates.sort_values(inplace=True, ascending=False)
final_recommendations = simCandidates.drop(userRatings.index, errors='ignore')

print(final_recommendations.head(50))

Indiana Jones and the Last Crusade (1989)                                         1.849580
Truman Show, The (1998)                                                           1.837368
E.T. the Extra-Terrestrial (1982)                                                 1.703163
Reservoir Dogs (1992)                                                             1.625913
Die Hard (1988)                                                                   1.483831
Twister (1996)                                                                    1.449790
Ocean's Eleven (2001)                                                             1.341948
Terminator, The (1984)                                                            1.326982
Goodfellas (1990)                                                                 1.307333
Saving Private Ryan (1998)                                                        1.263477
Princess Bride, The (1987)                                                        1.218119